# Starting with rag
## Requirements :
    -langchain-community
    -PyPDF
    -PymuPDF


### loading documents

In [ ]:
# Text Loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../doc_files/notes.txt")
content = loader.load()
print(content)


In [ ]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

dirLoader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs={"jq_schema": ".", "text_content": False},
    show_progress=False
)
content = dirLoader.load()
content

In [ ]:
# Loading Pdf File
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    loader_kwargs={"extract_images":False, "extract_tables": "markdown"},
    use_multithreading=True
)
contents = dir_loader.load()
contents

In [ ]:
# Create dummy files and write the some content 

import os

storage = {
    "notes.txt": "This is a simple text file content.",
    "config.json": '{"setting": "enabled", "version": 1.0}',
    "script.py": "print('Hello from the script!')",
}

for file, content in storage.items():
    with open(f"../doc_files/{file}", "w", encoding="utf-8") as f:
        f.write(content)


print("content Written Successfully...")

# RAG Pipeline (From Indexing to Vector Db pipleine)
### Requirements: 
    -langchain-community(PyPDFLoader and PyMuPDF)
    -langchain.textsplitter (RecurisveCharacterTextSplitter)
    pathlib

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
"""
create a function that loads all the pdf file in the dir and 
returns the whole documents by adding corresponding
metadata fields like file_name and filetype
"""
def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    print(pobj.rglob("**/*.pdf"))
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")

In [ ]:
# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    splitted_doc = text_splitter.split_documents(docs)
    print(f"splitted {len(docs)} Docs into {len(splitted_doc)}")
    # print(f"Content: {splitted_doc[0].page_content[:200]}")
    return splitted_doc
splitted_docs = split_doc(all_docs)
splitted_docs

# Embedding and vectordb

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np
import os

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model: {self.model_name} : {e}")
            raise ValueError("Model Cannot Be Loaded...")
        
    def generate_embedding(self, texts: List[str]):
        try:
            encodings = self.model.encode(texts, show_progress_bar=True)
            return encodings
        except Exception as e :
            print(e)
            
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model Doesn't Exist")
        return self.model.get_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

In [ ]:
# Vector Store
class VectorStore:
    def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.collection = None
        self.client = None
        self._load_vectorDB()
    
    # load the colleciton and client
    def _load_vectorDB(self):
        # initialize collection and client
        os.makedirs(self.persist_dir, exist_ok=True)
        try:
            self.client = chromadb.PersistentClient(path=self.persist_dir)
            self.client.delete_collection(name=self.collection_name)
            self.collection = self.client.get_or_create_collection(
                self.collection_name, 
                configuration={
                  "hnsw": {
                      "space": "cosine"
                  }
                },
                metadata={
                    "description": "PDF files for RAG"
                    }
                )
            print(f"Vector Store initialized successfully {self.collection_name}")
            print(f"Existing Documents in collection {self.collection.count()}")
        except Exception as e:
            print(e)    
    # Add docs to vector store 
    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        # prepare data 
        doc_ids = []
        doc_contents = []
        doc_embeddings = []
        metadatas = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            # unique id for each doc
            uid = f"{uuid.uuid4().hex[:8]}_{i}"
            doc_ids.append(uid)
            
            # prepare page content
            doc_contents.append(doc.page_content)
            
            # embeddings 
            doc_embeddings.append(embedding.tolist())
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            print(f"metadata: {metadata}")
            
        # add fields to vector Store 
        try:
            self.collection.add(
                ids=doc_ids,
                documents=doc_contents,
                embeddings=doc_embeddings,
                metadatas=metadatas
            )
            print(f"Successfully added {len(docs)} into vector store...")
        except Exception as e :
            print(e)


vectorstore_manager = VectorStore()

In [ ]:
# doc = Document(
#     page_content="Hello, LangChain!",
#     metadata={"source": "manual_input"}
# )
# emb = embedding_manager.generate_embedding([doc.page_content])
# emb.tolist()
# vectorstore_manager.add_docs(docs=[doc], embeddings=emb)


In [ ]:
# Extract the texts from page content 
texts = [doc.page_content for doc in splitted_docs]

# Generate Embedding 
embeddings = embedding_manager.generate_embedding(texts)

# store embedding and chunks into vector store 
vectorstore_manager.add_docs(splitted_docs, embeddings)

In [31]:
# Reterival Pipeline
class ReterivalManager:
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
    
    def reteriveContext(self, query: str, top_k: int, threshold: float = 0.0) -> list[Dict[str, Any]]:
        # Generate the qurey embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])
        
        # serach in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            reterived_docs = []
            if results["documents"] and results["documents"][0]:
                docs = results["documents"][0]
                ids = results["ids"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                
                for i, (id_, doc, metadata, distance) in enumerate(zip(ids, docs, metadatas, distances)):
                    # convert distanbce to simlarity score (Chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= threshold:
                        reterived_docs.append({
                            "id": id_,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Reterived {len(reterived_docs)}")
            else:
                print("no document Reterived ")
            return reterived_docs
        except Exception as e:
            print(e)

reterival = ReterivalManager(embedding_manager, vectorstore_manager)

In [ ]:
print(vectorstore_manager.collection.count())
print(vectorstore_manager.collection.configuration)
print(vectorstore_manager.collection.get())
print(vectorstore_manager.collection.count())

In [ ]:
results = vectorstore_manager.collection.get(
    include=["documents", "metadatas"]
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"], results["metadatas"])
):
    print(f"\n--- CHUNK {i} ---")
    print(doc)
    print("METADATA:", metadata)

In [ ]:
results = reterival.reteriveContext(
    query="Conclusion of the Srinivas Premier League Season ",
    top_k=10
)

for result in results:
    print(
        f"\nRank: {result['rank']}"
        f"\nSimilarity: {result['similarity_score']}"
        f"\nContent: {result['content'][:200]}"
    )

In [ ]:
results = reterival.reteriveContext(
    query="what are the skills of vivekanand",
    top_k=10
)

for result in results:
    print(
        f"\nRank: {result['rank']}"
        f"\nSimilarity: {result['similarity_score']}"
        f"\nContent: {result['content'][:200]}"
    )

In [43]:
# llm
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0.0, max_tokens=1024, timeout=60, max_retries=2)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021E65D09E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021E65D0AC10>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), request_timeout=60.0, max_tokens=1024)

In [30]:
# Create Enhanced RAG pipeline 

# Advance RAG Pipeline 
    - Citations
    - Streaming
    - History
    - Summarization

In [ ]:
class AdvanceRAG:
    def __init__(self, reteriver: ReterivalManager, llm):
        self.retriveer = reteriver
        self.llm = llm
        self.history = []
    
    # Query functrion to produce answer from retrieved docs
    def query_func(self, query: str, top_k: int, min_score: float = 0.0, stream: bool = False, summarize: bool = False):
        # retrive the document
        retrived_doc = self.retriveer.reteriveContext(query=query, top_k=top_k, threshold=min_score)
        if not retrived_doc:
            print("No Context Found ")
            context = ""
            sources = []
            answer = "No relevant chunk found"
        else:
            # prepare context along with citation label
            context = "\n\n".join(
                f"[{i}]\n{doc['content']}" for i, doc in enumerate(retrived_doc, start=1)
            )
            
            # prepare sources
            sources = [{
                "citation_id": i,
                "source": doc["metadata"].get("source", doc["metadata"].get("source_file", "unknown")),
                "page": doc["metadata"].get("page", "unknown"),
                "score": doc["similarity_score"],
                "preview": doc["content"][:120] + "..."
            } for i, doc in enumerate(retrived_doc,  start=1) ]
            
            prompt = f"""
                You are a RAG assistant.
                Answer the question using ONLY the provided context.
                Rules:
                - Do not use outside knowledge.
                - Do not invent information.
                - If the answer is not present in the context,
                say "I don't have enough information in the provided documents."
                - Add the citation number [1], [2], etc. after the
                statement supported by that context.
                Context:
                {context}
                Question:
                {query}
            """
            # stream the resp (if Stream)   
            if stream:
                pass
            response = self.llm.invoke([prompt])
            
            # citations(answer with citatiosn)
            
            # Summarize 
            
            # Store History(question, answer, sources, summary) 
            
            # return {'question', 'answer', 'sources', 'summary', 'history'}
        
        
rag = AdvanceRAG(llm=llm, reteriver=reterival)
rag

In [51]:
resp = rag.query_func("how to cook rice", 5)

Batches: 100%|██████████| 1/1 [00:00<00:00, 32.16it/s]


Reterived 5


In [52]:
resp.pretty_print()

================================== Ai Message ==================================

I don't have enough information in the provided documents.


In [ ]:
result = rag.query_func(
    query="what are the skills of vivekanand?",
    top_k=5
)